![JupyterLite HIV Drug Resistance Maps: an original conceptual globe, map, and molecular illustration.](assets/hivdr-splash.png)

**[Open this notebook in JupyterLite Lab](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/lab/index.html?path=10_country_study_matrix.ipynb)** · **[Read the Jupyter Book](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/book/)** · [Map and globe resources](mapping_resources.md)

Original conceptual artwork; its markers and bars are not observations.

# Country study matrix: eight domains

Project: JupyterLite HIV Drug Resistance Maps. Created/updated: 2026-11-16.
Author: [Repository contributors; generated with Codex]. Email/organization: [Not supplied].
Input: factual HIVDB metadata snapshot in `data/studies/`, plus the source-checked annotation ledger.
Output: computed pandas tables and Plotly figures; optional exports under `outputs/`.
Runtime: Python 3.11+, pandas, Plotly 6.2.0, IPython; native retrieval dependencies are listed in `study_query.py`.
Change log: initial reproducible study-inventory notebook. No simulated study data.

**Unit:** a distinct HIVDB reference/publication ID. A publication can describe multiple study components; multiple publications can describe one cohort. This is a descriptive inventory, not a population sample, national resistance estimate, or deduplicated systematic review.

The main matrix uses source metadata and nine source-checked publication annotations. **Age, baseline ART status, primary outcomes, clinical outcomes and study design are not complete structured fields in the selected exports.** Missing information stays visible. A second discovery matrix uses title cues, separately labelled as unverified.

The eight requested domains appear below; population is split into age and baseline ART status. A study can contribute to multiple categories. A zero means no matching publication in this snapshot/category, not no resistance or no research in that country. The primary-outcome field is deliberately separate from a paper's mutation/prevalence/susceptibility focus.

In [ ]:
# Read immutable factual inputs and check their recorded hashes before analysis.
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Code, HTML, FileLink
from study_matrix import (load_snapshot, country_membership, build_domains,
                          country_matrix, coverage_table, country_summary,
                          evidence_ledger, DOMAINS, UNKNOWN)
catalogue, evidence, signals, annotations, manifest = load_snapshot()
px.defaults.template = "plotly_white"
print("Snapshot built:", manifest["built_utc"])
print("Catalogue publication records:", len(catalogue))

In [ ]:
# Construct a long evidence table, then count DISTINCT references per country/category.
facts = build_domains(catalogue, evidence, annotations)
matrix = country_matrix(facts)
# This is the core matrix operation; repeated genes, samples and source rows do not multiply a publication.
transparent_matrix = facts.pivot_table(index=["iso3", "country"], columns=["domain", "value"],
    values="ref_id", aggfunc="nunique", fill_value=0).astype(int).sort_index()
pd.testing.assert_frame_equal(matrix, transparent_matrix)
print("Country/area and unmapped rows:", len(matrix), "| Domain/category columns:", len(matrix.columns))
display(coverage_table(facts, catalogue))

## Matrix table: information available by country and domain

Cells count distinct publications with at least one supported value for that domain. This compact view shows the 15 countries with the most represented publications. The export below contains every country and every category, including unknowns. It is a semantic pandas HTML table, so values can be selected and copied.

In [ ]:
# Rank countries only for display; all rows remain in the exported matrix.
summary = country_summary(catalogue, evidence)
top_countries = summary.nlargest(15, "Publications").country.tolist()
known = facts.loc[facts.value.ne(UNKNOWN) & facts.iso3.ne("")]
coverage_by_country = known.pivot_table(index="country", columns="domain", values="ref_id", aggfunc="nunique", fill_value=0)
coverage_by_country = coverage_by_country.reindex(columns=list(DOMAINS), fill_value=0).astype(int)
short_names = {"age_group":"Age", "art_status":"Baseline ART", "geography":"Geography", "primary_outcome":"Primary outcome",
               "drug_exposure":"Drug exposure", "clinical_outcome":"Clinical", "collection_year":"Sample year", "resistance_outcome":"Resistance", "study_design":"Design"}
display(coverage_by_country.reindex(top_countries, fill_value=0).rename(columns=short_names).style
        .background_gradient(cmap="Blues", axis=None).format("{:d}")
        .set_caption("Distinct publications with information, by country and requested domain"))

## Category matrix: compare study types

This view displays baseline ART status and study/sampling design for the countries represented in the nine source-checked annotations. These annotations are a small worked extraction set, not an exhaustive review. Randomized trials can also be longitudinal; category totals therefore overlap. `annotations.csv` retains source links, cohort labels, date, extraction status and limitations.

In [ ]:
# Include every country linked to an annotated publication; retain unknown categories.
annotated_countries = set(facts.loc[facts.status.eq("Source-checked annotation") & facts.iso3.ne(""), "country"])
type_matrix = country_matrix(facts, ["art_status", "study_design"], mapped_only=True)
display(type_matrix.loc[type_matrix.index.get_level_values("country").isin(annotated_countries)].style
        .format("{:d}").set_caption("Country × baseline ART / study-design categories (source-checked subset)"))

## Follow a cell back to publications

Edit the country, domain and optional category below. The returned reference IDs are the audit trail for a matrix cell. Publication-level classifications apply to the study report; they do not establish that every clinical outcome was measured in every country subgroup.

In [ ]:
# Reproducible drill-down parameters: change and rerun this cell.
COUNTRY_ISO3 = "NGA"
DOMAIN = "study_design"
CATEGORY = None  # e.g. "Cross-sectional"; None retains known and missing values.
selected = facts.loc[facts.iso3.eq(COUNTRY_ISO3) & facts.domain.eq(DOMAIN)]
if CATEGORY is not None:
    selected = selected.loc[selected.value.eq(CATEGORY)]
audit = selected.merge(catalogue[["ref_id","author_year","publication_year","pmid","source_url"]], on="ref_id", validate="many_to_one")
display(audit[["ref_id","author_year","value","status","source","source_url"]].drop_duplicates().reset_index(drop=True))
print("Distinct publications in this selection:", audit.ref_id.nunique())

## Mutation, prevalence and susceptibility: outcomes studied versus primary endpoints

The phenotype-reference flag only means that a reference occurs in Stanford's genotype-phenotype reference export. It does not establish a clinical study's primary endpoint or the country of the laboratory assay. The small source-checked outcome-focus table is provided below. For example, SAILING's primary endpoint is viral suppression even though resistance is also studied.

In [ ]:
# Show publication-level factual annotations and provenance, without assigning endpoint status from a title.
focus = annotations.loc[annotations.domain.isin(["primary_outcome","outcome_focus"])]
display(focus[["ref_id","cohort_label","domain","value","source_url"]].reset_index(drop=True))

## Discovery matrix: unverified title cues

These transparent regular-expression matches help prioritize full-text extraction. A title can mention a population or outcome in the background rather than the actual study. In particular, a “DTG” title match does not establish DTG exposure, and a suppression title match does not identify the denominator. **Do not use these cues for meta-analysis eligibility without verification.**

In [ ]:
# Keep discovery cues in a different object, with a different label, from source-supported facts.
membership = country_membership(catalogue, evidence)
cues = membership[["ref_id","iso3","country"]].merge(signals, on="ref_id", how="inner", validate="many_to_many")
cue_matrix = cues.loc[cues.iso3.ne("") & cues.domain.eq("study_design")].pivot_table(
    index="country", columns="value", values="ref_id", aggfunc="nunique", fill_value=0).astype(int)
display(cue_matrix.reindex(top_countries, fill_value=0).style.format("{:d}")
        .set_caption("UNVERIFIED title cues — publications to screen, not confirmed study types"))

## Export the reproducible results

The full category matrix, long evidence ledger, one-row-per-publication/country table and separate discovery matrix can be downloaded from the Lab file browser. Run this notebook to recreate them. All unknown and unmapped rows are retained.

In [ ]:
# Use package-native CSV and semantic HTML exports, generated from the same computed objects.
out = Path("outputs/study_matrix"); out.mkdir(parents=True, exist_ok=True)
matrix.to_csv(out / "country_domain_matrix.csv")
matrix.style.format("{:d}").set_caption("Distinct publication counts; unknowns retained").to_html(out / "country_domain_matrix.html")
facts.to_csv(out / "domain_evidence_long.csv", index=False)
evidence_ledger(catalogue, evidence, annotations).to_csv(out / "publication_country_ledger.csv", index=False)
cue_matrix.to_csv(out / "UNVERIFIED_title_cues.csv")
for path in sorted(out.glob("*")):
    print(path.as_posix())

Continue to the [visual atlas](11_study_evidence_atlas.ipynb), then **Meta Analysis** for the Africa-focused evidence synthesis. Read [source and field documentation](study_matrix_methods.md) before extending the classifications.